> 在线讲义：[第 39 章 · 跨节点流量与物理路径：同样的字节，走哪几张网卡](https://wikieden.github.io/ai-infra-from-constraints/lessons/39-cross-node.html) · 本书《从约束推导 AI Infra》

# 第 39 章 · 跨节点流量与物理路径：同样的字节，走哪几张网卡

配套讲义：仓库里 `ai-infra/notes/lessons/39-cross-node.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



例题 7.2 三种归约、对照 PCIe 配置、在网归约、层间交接、EP 接收、多网卡中继取自 AII 第 7 章 §7.2。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
MiB, GiB = 2**20, 2**30
NIC, NVL, A = 50e9, 450e9, 0.83e-6

**函数 `ring_continuous_ms`**：连续环：每轮等跨服务器那条边。

In [ ]:
def ring_continuous_ms(M: float = 192 * MiB, p: int = 16, bw: float = NIC) -> float:
    """连续环：每轮等跨服务器那条边。"""
    return 2 * (p - 1) * (A + M / p / bw) * 1e3

**函数 `hierarchical_ms`**：分层：本地 RS + AG 共 2(local-1) 轮，跨服务器 2(servers-1) 轮。

In [ ]:
def hierarchical_ms(M: float = 192 * MiB, local: int = 8, servers: int = 2, BL: float = NVL, BN: float = NIC) -> float:
    """分层：本地 RS + AG 共 2(local-1) 轮，跨服务器 2(servers-1) 轮。"""
    shard = M / local
    t_local = 2 * (local - 1) * shard / BL
    t_remote = 2 * (servers - 1) * shard / servers / BN
    return (t_local + t_remote + (2 * (local - 1) + 2 * (servers - 1)) * A) * 1e3

**函数 `cross_bytes_mib`**：小工具。

In [ ]:
def cross_bytes_mib() -> tuple[float, float, float]:
    return 2 * 30 * 12, 16 * 30 * 12, 8 * 48

**函数 `in_network_ms`**：跨服务器阶段：(环, 在网归约)。

In [ ]:
def in_network_ms(S: int, shard: float = 24 * MiB) -> tuple[float, float]:
    """跨服务器阶段：(环, 在网归约)。"""
    ring = 2 * (S - 1) * A + 2 * (S - 1) / S * shard / NIC
    inn = A + shard / NIC
    return ring * 1e3, inn * 1e3

**函数 `msg_us`**：小工具。

In [ ]:
def msg_us(M: float, alpha: float = 5e-6, bw: float = NIC) -> float:
    return (alpha + M / bw) * 1e6

**函数 `relay_gbps`**：小工具。

In [ ]:
def relay_gbps(BD: float) -> float:
    return min(BD, 50 + min(450, 100))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：例题 7.2，192 MiB 梯度，两台 HGX H100")
print(f"  跨服务器字节：连续 / 交错 / 分层 = {cross_bytes_mib()} MiB")
print(f"  连续环 {ring_continuous_ms():.1f} ms；分层 {hierarchical_ms():.2f} ms")
print(f"  分层合算的本地带宽下限 ≈ {336 * MiB / ((ring_continuous_ms() - 0.503) * 1e-3) / 1e9:.0f} GB/s")
ring8 = 14 * 24 * MiB / 32e9 * 1e3
loc, rem = 6 * 48 * MiB / 32e9 * 1e3, 2 * 96 * MiB / 32e9 * 1e3
print(f"  对照 PCIe 配置：连续环 {ring8:.1f} ms，分层 {loc:.1f} + {rem:.1f} = {loc + rem:.1f} ms；本地 300 GB/s 时 {6 * 48 * MiB / 300e9 * 1e3 + rem:.1f} ms")
print(f"  持平的本地带宽 ≈ {6 * 48 * MiB / ((ring8 - rem) * 1e-3) / 1e9:.0f} GB/s")
for S in (2, 8, 32):
    r, i = in_network_ms(S)
    print(f"  在网归约 {S:2d} 台：环 {r:.3f} ms，在网 {i:.3f} ms")

In [ ]:
print("\n第二步：TP / PP 交接，α = 5 μs")
print(f"  64 MiB {msg_us(64 * MiB) / 1e3:.2f} ms，8 KiB {msg_us(8192):.2f} μs；转折 {5e-6 * NIC / 1024:.0f} KiB")
print(f"  4 阶段 8 micro-batch：{8 + 3} ms，利用率 {8 / 11:.0%}；90% 要 {next(m for m in range(1, 100) if m / (m + 3) >= 0.9)} 个")

In [ ]:
print("\n第三步：EP，1024 token × 8 专家 × 8 KiB，一半跨服务器")
v = 1024 * 8 * 8 * 1024 * 0.5
print(f"  dispatch {v / MiB:.0f} MiB：一张网卡 {v / NIC * 1e3:.2f} ms，八张 {v / 8 / NIC * 1e3:.3f} ms，共享 32 GB/s 插槽 {v / 32e9 * 1e3:.2f} ms")

In [ ]:
print("\n第四步：多网卡")
print(f"  对照配置单端口 {14 * 24 * MiB / 25e9 * 1e3:.1f} ms，双端口受插槽限制 {14 * 24 * MiB / 32e9 * 1e3:.1f} ms")
for BD in (50, 100, 150, 200):
    print(f"  下游 {BD} GB/s：可用 {relay_gbps(BD)} GB/s，1.125 GiB {1.125 * GiB / (relay_gbps(BD) * 1e9) * 1e3:.1f} ms")
print(f"  δ = 18.6 ns 时速率与带宽交叉的消息大小 {NIC * 18.6e-9:.0f} B；192 MiB 一张网卡 {192 * MiB / NIC * 1e3:.2f} ms")

## 3. 代码里才看得到的

1. `hierarchical_ms` 的本地部分：8 个参与者 RS + AG 共 14 轮，每轮每人发 $M/8 = 24$ MiB。
2. 对照配置的分层：4 卡 RS 3 轮 + AG 3 轮，每轮 48 MiB；跨服务器 2 轮，每轮 96 MiB 都挤过一个插槽。
3. 在网归约只计网卡发送与一次启动，交换机归约引擎的吞吐另作输入。

## 4. 自己改一改

- 给 `hierarchical_ms` 加服务器数参数，画 2、4、8 台的时间。
- 写一个流量矩阵函数：输入 $V_{ij}$ 和每卡网卡带宽，输出最慢的发送方、接收方和割集。
- 把 `relay_gbps` 改成 NVLink 同时承担本机归约、只剩 60 GB/s 给中继的情况。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 排一圈只走两扇门，交错走全门却运八倍，先楼内合并再八门各运一份；
> 大箱看卡车，小螺丝全在开门签收；
> 分拣中心多开口，后面共一条传送带还是卡；
> 车道要通到外面，还要有车分过去；同号车道不上主干道。